In [0]:
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.subscription_events")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")

item_ids_search = [
    "28000",
    "250131",
    "22024",
    "29324",
    "250133",
    "28071",
    "28027",
    "40598",
    "28034",
    "28144",
    "28281",
    "40423",
    "36028",
    "22115",
    "28049",
    "28423",
    "28412",
    "33059",
    "28424"
]


In [0]:
from pyspark.sql import functions as F

df_master = spark.table("data_exploration_hub.marketing_hub.df_master_transactions")

In [0]:
df_query = (T_trans.join(T_items, T_trans.ItemSkey == T_items.itemskey, "left")
            .join(T_date.select("dateid", "datefull"), T_trans.TransactionDateid == T_date.dateid, "left")
            .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
.filter(
    (F.col("Item_ID").isin(
    "28000", "250131", "22024", "29324", "250133",
    "28071", "28027", "40598", "28034", "28144",
    "28281", "40423", "36028", "22115", "28049",
    "28423", "28412", "33059", "28424")) & 
    (F.col("datefull").between("2025-01-01", "2025-01-31"))
)
.groupBy("CDE_ID")
.agg((F.sum("Quantity").alias("Total Quantity Sold")),
      F.sum(
              F.when(F.col("SceneFLAG") == "Yes", 1).otherwise(0)
          ).alias("Scene_Transactions"),
     F.sum(
         F.when(F.col("SceneFLAG") == "Yes", F.col("Net_Revenue"))
     ).alias("Total Scene Revenue"),
     F.sum(
         F.when(F.col("SceneFLAG") == "No", F.col("Net_Revenue"))
     ).alias("Total Non Scene Revenue"),

     F.sum(
              F.when(F.col("SceneFLAG") == "No", 1).otherwise(0)
          ).alias("Non_Scene_Transactions"),
      )
.filter(F.col("CDE_ID").isNotNull())
)




In [0]:
display(df_query.select("CDE_ID").distinct().count())